In [1]:
!pip install torch
!pip install scikit-learn


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import networkx as nx
from packaging import version
import sys
import numpy as np
import sklearn
import torch

print("Python version:", sys.version)
print("networkx version:", nx.__version__)
print("sklearn version:", sklearn.__version__)
print("torch version:", torch.__version__)

assert version.parse(nx.__version__) >= version.parse("2.6")
assert version.parse(torch.__version__) >= version.parse("1.10.0")
assert version.parse(sklearn.__version__) >= version.parse("1.1.3")
assert sys.version_info[0] == 3
assert sys.version_info[1] >= 9

try:
    ipython = get_ipython()
except NameError:
    ipython = None

if ipython is not None and 'google.colab' in str(ipython):
    print('Working in colab')
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print("working locally")

Python version: 3.11.11 (main, Mar 14 2026, 01:14:19) [GCC 12.2.0]
networkx version: 3.6.1
sklearn version: 1.8.0
torch version: 2.11.0+cu130
working locally


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class GCN(nn.Module):
    def __init__(self, d_in, d_hidden, n_classes, dropout=0.5):
        super().__init__()
        self.lin1 = nn.Linear(d_in, d_hidden)
        self.lin2 = nn.Linear(d_hidden, n_classes)
        self.dropout = dropout

    def forward(self, x, A_hat):
        # Première couche : convolution + ReLU
        h = F.relu(A_hat @ self.lin1(x))
        # Dropout pour la régularisation
        h = F.dropout(h, p=self.dropout, training=self.training)
        # Deuxième couche : convolution finale
        return A_hat @ self.lin2(h)

In [4]:
def train(optimizer, model, x, A_hat, y, train_mask, val_mask, device="cpu"):
    # Déplacer les données vers le device spécifié
    x = x.to(device)
    A_hat = A_hat.to(device)
    y = y.to(device)
    train_mask = train_mask.to(device)
    val_mask = val_mask.to(device)

    model.train()
    optimizer.zero_grad()

    # Forward pass
    logits = model(x, A_hat)
    # Calcul de la perte uniquement sur les nœuds d'entraînement
    loss = F.cross_entropy(logits[train_mask], y[train_mask])
    # Perte de validation
    val_loss = F.cross_entropy(logits[val_mask], y[val_mask])

    loss.backward()
    optimizer.step()
    return float(loss), float(val_loss)


def accuracy(pred, target, mask):
    correct = (pred[mask] == target[mask]).float()
    return float(correct.mean()) if correct.numel() > 0 else 0.0

In [5]:
from sklearn.model_selection import train_test_split
ALL_ATTRS = ["student_fac", "gender", "major_index", "dorm", "year", "second_major", "high_school"]

def build_A_hat(graph, label_attr):
    """
    Construit la matrice d'adjacence normalisée et les features pour le graphe.
    """
    # Ordre des nœuds stable
    node_order = sorted(graph.nodes(), key=lambda node: int(node))

    # Vérifier la présence du label parmi les attributs; sinon on continue quand même
    feature_attrs = [a for a in ALL_ATTRS if a != label_attr]

    # Construire les one-hot pour chaque attribut catégoriel
    feats = []
    for attr in feature_attrs:
        vals = [graph.nodes[node].get(attr, 0) for node in node_order]
        uniq = sorted(set(vals))
        col = {x: i for i, x in enumerate(uniq)}
        oh = np.zeros((len(node_order), len(uniq)), dtype=np.float32)
        for i, x in enumerate(vals):
            oh[i, col[x]] = 1.0
        feats.append(oh)

    # Ajouter la feature 'degré' normalisée
    deg = np.array([graph.degree(node) for node in node_order], dtype=np.float32)
    deg = (deg - deg.mean()) / (deg.std() + 1e-6)
    feats.append(deg.reshape(-1, 1))

    # Concaténer et convertir en tensor
    x = torch.tensor(np.concatenate(feats, axis=1), dtype=torch.float)

    # Construire le vecteur de labels puis remapper les classes valides à 0..K-1
    y_raw = [graph.nodes[node].get(label_attr) for node in node_order]
    y_raw = [(-1 if v is None else v) for v in y_raw]
    # Identifier classes valides et construire mapping
    valid_vals = [v for v in y_raw if v != -1]
    if len(valid_vals) > 0:
        classes = sorted(set(int(v) for v in valid_vals))
        mapping = {c: i for i, c in enumerate(classes)}
        y_np = np.array([mapping[int(v)] if v != -1 else -1 for v in y_raw], dtype=np.int64)
    else:
        y_np = np.array([-1 for _ in y_raw], dtype=np.int64)
    y = torch.tensor(y_np, dtype=torch.long)

    # Matrice d'adjacence (avec self-loops) et normalisation symétrique
    adj = nx.to_numpy_array(graph, nodelist=node_order, dtype=np.float32)
    adj = adj + np.eye(adj.shape[0], dtype=np.float32)
    deg = adj.sum(axis=1)
    deg_inv_sqrt = np.power(deg, -0.5)
    deg_inv_sqrt[deg == 0] = 0.0 ## pour éviter les divisions par zéro
    A_hat = deg_inv_sqrt[:, None] * adj * deg_inv_sqrt[None, :]
    A_hat = torch.tensor(A_hat, dtype=torch.float)

    return x, y, A_hat, node_order, feature_attrs

In [6]:
from tqdm import tqdm


def split_indices(indices, seed):
    rng = np.random.default_rng(seed)
    indices = np.array(indices, dtype=int)
    rng.shuffle(indices)
    # On sépare les indices en 70% train, 10% validation, 20% test
    n_val = max(1, int(round(0.1 * len(indices))))
    n_test = max(1, int(round(0.2 * len(indices))))
    test_idx = indices[:n_test]
    val_idx = indices[n_test:n_test + n_val]
    train_idx = indices[n_test + n_val:]
    return train_idx, val_idx, test_idx


def train_GCN_for_recovery(graph, label_attr, nodes_to_remove, device='cpu', lr=0.001, n_epoch=100, emb_size=50, dp=0.5):
    """
    Entraîne un modèle GCN pour prédire les labels de nœuds supprimés.
    """
    # Construire les données (features, labels, matrice d'adjacence)
    x, y, A_hat, node_order, feature_attrs = build_A_hat(graph, label_attr)
    N = x.shape[0]

    # Mapper les indices de nœuds à supprimer
    nodes_to_remove_idx = [node_order.index(n) for n in nodes_to_remove]
    nodes_to_remove_idx = np.array(nodes_to_remove_idx, dtype=int)

    labels = y.numpy()

    # Indices des nœuds avec labels (ceux non supprimés)
    labeled_idx = np.array([i for i in range(N) if i not in nodes_to_remove_idx and labels[i] >= 0], dtype=int)

    # Séparer les nœuds étiquetés en train / validation / test
    train_idx, val_idx, test_idx = split_indices(labeled_idx, seed=0)

    # Créer les masques pour train/val/test
    train_mask = torch.zeros(N, dtype=torch.bool)
    val_mask = torch.zeros(N, dtype=torch.bool)
    test_mask = torch.zeros(N, dtype=torch.bool)
    train_mask[torch.as_tensor(train_idx)] = True
    val_mask[torch.as_tensor(val_idx)] = True
    test_mask[torch.as_tensor(test_idx)] = True

    # Déplacer les données vers le device et initialiser le modèle
    x = x.to(device) # Features des noeuds
    A_hat = A_hat.to(device) # 
    y = y.to(device) # labels
    model = GCN(x.shape[1], emb_size, int(y.max().item()) + 1, dropout=dp).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Entraîner le modèle
    for epoch in range(n_epoch):
        loss, _ = train(optimizer, model, x, A_hat, y, train_mask, val_mask, device=device)

    # Évaluer sur les nœuds supprimés
    logits = model(x, A_hat).detach().cpu()
    preds = logits.argmax(dim=-1).numpy()

    # Récupérer les vraies labels et prédictions pour les nœuds supprimés
    true = labels[nodes_to_remove_idx]
    pred = preds[nodes_to_remove_idx]

    # Filtrer les nœuds avec labels valides (non -1)
    valid = true != -1
    if valid.sum() == 0:
        return 1.0, 0.0

    true_valid = true[valid]
    pred_valid = pred[valid]

    # Calculer les métriques
    acc = float(np.mean(true_valid == pred_valid))
    mae = float(np.mean(np.abs(true_valid.astype(float) - pred_valid.astype(float))))
    return mae, acc


def evaluate_gcn_recovery(graph, attr_name, fractions_removed=[0.1, 0.2, 0.3, 0.4], n_trials=5, device='cpu'):
    """
    Évalue la récupération de labels pour différentes fractions de nœuds supprimés.
    """
    results = {}
    nodes = list(graph.nodes())
    
    # Pour chaque fraction de nœuds à supprimer
    for frac in fractions_removed:
        maes = [] # moyenne d'erreur
        accs = [] # précision
        
        # Effectuer plusieurs trials
        for trial in tqdm(range(n_trials), desc=f"  Trials pour frac={frac}"):
            n_to_remove = max(1, int(len(nodes) * frac))
            nodes_to_remove = list(np.random.choice(nodes, size=n_to_remove, replace=False))
            mae, acc = train_GCN_for_recovery(graph, attr_name, nodes_to_remove, device=device)
            maes.append(mae)
            accs.append(acc)
        
        results[frac] = (float(np.mean(maes)), float(np.mean(accs)))
        print(f"Fraction supprimée: {frac:.2f} | MAE moyen: {results[frac][0]:.4f} | Accuracy moyen: {results[frac][1]:.4f}")
    return results

In [7]:
import pandas as pd
from tqdm import tqdm

# Paramètres d'évaluation
fractions = [0.1, 0.2, 0.3, 0.4]
n_trials = 7
device = "cpu"
results_table_mae = {}
results_table_acc = {}

# Charger le réseau et initialiser
network_file = "data/MIT8.gml"
G_recovery = nx.read_gml(network_file)
attrs = ["dorm", "major_index", "gender"]

# Évaluer chaque attribut avec une barre de progression
for attr in attrs:
    print(f"\nAttributs en cours: {attr}")
    results = evaluate_gcn_recovery(G_recovery, attr, fractions_removed=fractions, n_trials=n_trials, device=device)
    # Extraire les MAE et précisions
    results_table_mae[attr] = {frac: results[frac][0] for frac in fractions}
    results_table_acc[attr] = {frac: results[frac][1] for frac in fractions}

# Afficher résultats de précision
df_acc = pd.DataFrame(results_table_acc).T # sympa pour faire des tableaux
df_acc.columns = [f"{int(frac*100)}%" for frac in fractions]
print("\n"+ "-" * 60)
print("TABLE 1: ACCURACY SCORE")
print("-" * 50)
print(df_acc.round(3))

# Afficher résultats d'erreur absolue moyenne
df_mae = pd.DataFrame(results_table_mae).T
df_mae.columns = [f"{int(frac*100)}%" for frac in fractions]
print("\n" + "-" * 60)
print("TABLE 2: MEAN ABSOLUTE ERROR")
print("-" * 60)
print(df_mae.round(3))


Attributs en cours: dorm


  Trials pour frac=0.1:   0%|          | 0/7 [00:00<?, ?it/s]/tmp/ipykernel_22000/2216656796.py:21: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  return float(loss), float(val_loss)
  Trials pour frac=0.1: 100%|██████████| 7/7 [01:26<00:00, 12.29s/it]


Fraction supprimée: 0.10 | MAE moyen: 13.0830 | Accuracy moyen: 0.2482


  Trials pour frac=0.2: 100%|██████████| 7/7 [01:30<00:00, 12.96s/it]


Fraction supprimée: 0.20 | MAE moyen: 12.9587 | Accuracy moyen: 0.2423


...

Fraction supprimée: 0.20 | MAE moyen: 0.5303 | Accuracy moyen: 0.5658


  Trials pour frac=0.3: 100%|██████████| 7/7 [02:21<00:00, 20.25s/it]


Fraction supprimée: 0.30 | MAE moyen: 0.5206 | Accuracy moyen: 0.5686


  Trials pour frac=0.4: 100%|██████████| 7/7 [02:10<00:00, 18.68s/it]

Fraction supprimée: 0.40 | MAE moyen: 0.5214 | Accuracy moyen: 0.5692

------------------------------------------------------------
TABLE 1: ACCURACY SCORE
--------------------------------------------------
               10%    20%    30%    40%
dorm         0.248  0.242  0.239  0.244
major_index  0.211  0.216  0.217  0.223
gender       0.570  0.566  0.569  0.569

------------------------------------------------------------
TABLE 2: MEAN ABSOLUTE ERROR
------------------------------------------------------------
                10%     20%     30%     40%
dorm         13.083  12.959  12.851  12.954
major_index   5.592   5.626   5.590   5.548
gender        0.516   0.530   0.521   0.521
